# ArcticShift Regression Analysis
Runs all six regression models from Chapter 4 of the methodology on the combined
ArcticShift lexical CSV produced by `combine_lexical_csvs.py`.

In [ ]:
import warnings
import pandas as pd

from regressions import (
    run_baseline_ols,
    run_first_diff_ols,
    run_ar_ols,
    run_cross_user_wls,
    run_fixed_effects_panel,
    run_mixed_effects,
    LEXICAL_METRICS,
)

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.6f}".format)

# ---------------------------------------------------------------------------
# Configuration
# ---------------------------------------------------------------------------
# Path to the combined CSV written by combine_lexical_csvs.py.
# Edit this to match your local or cluster path before running.
CSV_PATH = "/scratch/network/nv9344/Thesis/Thesis-Data/ArcticShift/lexical_df_combined.csv"

METRICS  = LEXICAL_METRICS   # all six, or pass a subset to any function below
ALPHA    = 0.05              # significance level
APPLY_BH = True              # Benjamini-Hochberg FDR correction

In [ ]:
# Load data
# raw_text is excluded via usecols — not needed for any regression
COLS_TO_LOAD = [
    "utterance_id", "speaker_id", "subreddit",
    "timestamp", "year_month",
    "num_utterances_by_speaker", "num_utterances_by_speaker_month",
    "log_freq_month",
    "post_depth", "score", "num_direct_replies", "controversiality", "edited",
    "mtld_score", "mattr_score", "yules_k", "zipf_score", "aoa_score", "nawl_ratio",
]

df = pd.read_csv(CSV_PATH, usecols=COLS_TO_LOAD, low_memory=False)

print(f"Loaded {len(df):,} utterances across {df['subreddit'].nunique()} subreddit(s).")
print(df[["subreddit", "year_month"]].groupby("subreddit")["year_month"].nunique()
        .rename("n_months").to_frame().T)

# Baseline OLS Regression
Model: `y_t = β₀ + β₁·t + ε_t` — one regression per (subreddit × metric), fitted on
monthly-aggregated data with Newey-West HAC standard errors. β₁ is the estimated
change in the metric per calendar month.

In [ ]:
ols_results = run_baseline_ols(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

# Summary counts
print("=== Baseline OLS — conclusion counts ===")
print(ols_results["conclusion"].value_counts().to_string())
print()

# Significant results
sig = ols_results[ols_results["significant"] == True].copy()
print(f"Significant (subreddit × metric) pairs: {len(sig)} / {len(ols_results)}")
if len(sig):
    print(sig[["subreddit", "metric", "beta_1", "se_beta_1", "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

ols_results

In [ ]:
from trend_analysis import plot_ols_trend_grid

agg = df.groupby(["subreddit", "year_month"])[METRICS].mean().reset_index()
plot_ols_trend_grid(agg, ols_results)

# First Differenced OLS Regression
Model: `Δy_t = β₀ + ε_t` — the drift model fitted on first-differenced monthly series
with HAC standard errors. A significant β₀ (drift) with the same sign as β₁ from the
baseline levels regression provides convergent evidence of a genuine trend.

In [ ]:
fd_results = run_first_diff_ols(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== First-Differenced OLS — conclusion counts ===")
print(fd_results["conclusion"].value_counts().to_string())
print()

sig_fd = fd_results[fd_results["significant"] == True].copy()
print(f"Significant pairs: {len(sig_fd)} / {len(fd_results)}")
if len(sig_fd):
    print(sig_fd[["subreddit", "metric", "drift", "se_drift", "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

# Convergence check: compare signs with baseline OLS
merged = ols_results[["subreddit", "metric", "beta_1", "significant"]].rename(
    columns={"beta_1": "ols_beta1", "significant": "ols_sig"}
).merge(
    fd_results[["subreddit", "metric", "drift", "significant"]].rename(
        columns={"drift": "fd_drift", "significant": "fd_sig"}
    ),
    on=["subreddit", "metric"],
)
merged["signs_agree"] = (
    merged["ols_beta1"].apply(lambda x: 1 if x > 0 else -1) ==
    merged["fd_drift"].apply(lambda x: 1 if x > 0 else -1)
)
print()
print("=== OLS ↔ First-Diff sign convergence ===")
print(merged[["subreddit", "metric", "ols_beta1", "ols_sig", "fd_drift", "fd_sig", "signs_agree"]]
      .sort_values(["metric", "subreddit"])
      .to_string(index=False))

fd_results

# AR OLS Regression
Model: `y_t = β₀ + β₁·t + φ·y_{t-1} + ε_t` — adds an AR(1) term to the baseline to
absorb autocorrelation, with HAC standard errors. β₁ here captures the trend net of
persistence; φ captures the autoregressive pull of the previous month.

In [ ]:
ar_results = run_ar_ols(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== AR OLS — conclusion counts ===")
print(ar_results["conclusion"].value_counts().to_string())
print()

sig_ar = ar_results[ar_results["significant"] == True].copy()
print(f"Significant pairs: {len(sig_ar)} / {len(ar_results)}")
if len(sig_ar):
    print(sig_ar[["subreddit", "metric", "beta_1", "se_beta_1", "phi",
                  "p_value_beta1", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

# Three-model sign convergence: OLS, FD, AR all agree → strong evidence
convergence = merged.merge(
    ar_results[["subreddit", "metric", "beta_1", "significant"]].rename(
        columns={"beta_1": "ar_beta1", "significant": "ar_sig"}
    ),
    on=["subreddit", "metric"],
)
convergence["all_agree"] = (
    convergence["signs_agree"] &
    (convergence["ols_beta1"].apply(lambda x: 1 if x > 0 else -1) ==
     convergence["ar_beta1"].apply(lambda x: 1 if x > 0 else -1))
)
print()
print("=== Three-model sign convergence (OLS, FD, AR) ===")
print(convergence[["subreddit", "metric", "ols_sig", "fd_sig", "ar_sig", "all_agree"]]
      .sort_values(["all_agree", "metric", "subreddit"], ascending=[False, True, True])
      .to_string(index=False))

ar_results

# Cross-User WLS Regression
Model: `ȳ_u = β₀ + β₁·F̄_u + β₂·X̄_u + ε_u` — user-level means, one regression per
(subreddit × metric), weighted by total post count n_u. β₁ captures the relationship
between posting frequency and lexical quality across users; β₂ is a vector of control
coefficients on user-mean post depth, edited status, score, number of direct replies,
and controversiality.

In [ ]:
wls_results = run_cross_user_wls(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== Cross-User WLS — conclusion counts ===")
print(wls_results["conclusion"].value_counts().to_string())
print()

sig_wls = wls_results[wls_results["significant"] == True].copy()
print(f"Significant pairs: {len(sig_wls)} / {len(wls_results)}")
if len(sig_wls):
    print(sig_wls[["subreddit", "metric", "n_users", "beta_1", "se_beta_1",
                   "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

wls_results

In [ ]:
# β₂ Control Coefficients — Cross-User WLS
# Each element of β₂ captures how user-mean contextual/engagement characteristics
# correlate with lexical quality, holding constant posting frequency (β₁).
print("=== Cross-User WLS — β₂ Control Coefficients ===\n")

_ctrl_names = ["post_depth", "edited", "score", "num_direct_replies", "controversiality"]
_b2_cols  = [f"beta2_{c}"    for c in _ctrl_names if f"beta2_{c}"    in wls_results.columns]
_se_cols  = [f"se_beta2_{c}" for c in _ctrl_names if f"se_beta2_{c}" in wls_results.columns]
_avail    = [c for c in _ctrl_names if f"beta2_{c}" in wls_results.columns]

if _b2_cols:
    _b2 = (
        wls_results[["subreddit", "metric"] + _b2_cols + _se_cols]
        .rename(columns={f"beta2_{c}": c for c in _avail}
                | {f"se_beta2_{c}": f"se_{c}" for c in _avail})
        .sort_values(["metric", "subreddit"])
    )
    print(_b2.to_string(index=False))
else:
    print("β₂ columns not found — re-run after updating regressions.py.")

# Fixed Effects Panel Regression
Model: `y_ust = β₁·F_ut + β₂·X_ust + α_u + γ_t + δ_s + ε_ust` — user (α_u), time (γ_t), and
subreddit (δ_s) fixed effects across all communities jointly. β₁ is the within-user effect
of log monthly posting frequency on lexical quality; β₂ captures the effects of time-varying
post-level controls (post depth, edited status, score, direct replies, controversiality).

In [ ]:
fe_results = run_fixed_effects_panel(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== Fixed Effects Panel — conclusion counts ===")
print(fe_results["conclusion"].value_counts().to_string())
print()

sig_fe = fe_results[fe_results["significant"] == True].copy()
print(f"Significant metrics: {len(sig_fe)} / {len(fe_results)}")
if len(sig_fe):
    print(sig_fe[["metric", "n_obs", "n_users", "n_periods",
                  "beta_1", "se_beta_1", "p_value", "p_value_bh", "conclusion"]]
          .to_string(index=False))

fe_results

In [ ]:
# β₂ Control Coefficients — Fixed Effects Panel
# Each element of β₂ captures within-user variation in post-level contextual features,
# conditional on user (α_u), time (γ_t), and subreddit (δ_s) fixed effects.
# One row per metric; β₂ is pooled across all subreddits in the joint panel.
print("=== Fixed Effects Panel — β₂ Control Coefficients ===\n")

_ctrl_names = ["post_depth", "edited", "score", "num_direct_replies", "controversiality"]
_b2_cols  = [f"beta2_{c}"    for c in _ctrl_names if f"beta2_{c}"    in fe_results.columns]
_se_cols  = [f"se_beta2_{c}" for c in _ctrl_names if f"se_beta2_{c}" in fe_results.columns]
_avail    = [c for c in _ctrl_names if f"beta2_{c}" in fe_results.columns]

if _b2_cols:
    _b2 = (
        fe_results[["metric"] + _b2_cols + _se_cols]
        .rename(columns={f"beta2_{c}": c for c in _avail}
                | {f"se_beta2_{c}": f"se_{c}" for c in _avail})
    )
    print(_b2.to_string(index=False))
else:
    print("β₂ columns not found — re-run after updating regressions.py.")

# Cross-Subreddit Mixed-Effects Comparison
Model: `y_ust = Σ_s θ_s·1[subreddit=s] + β₁·F_ut + β₂·X_ust + a_u + γ_t + ε_ust` — subreddit
fixed effects (θ_s) with user random intercepts (a_u), a frequency term (β₁·F_ut), time fixed
effects (γ_t), and post-level controls (β₂·X_ust), fitted via REML. θ_s is the conditional
mean difference in lexical quality for subreddit s relative to the reference (first
alphabetically); β₂ captures the same post-level control effects as in the FE model.

In [ ]:
mixed_results = run_mixed_effects(
    df,
    metrics=METRICS,
    alpha=ALPHA,
    apply_bh=APPLY_BH,
)

print("=== Mixed Effects — conclusion counts ===")
print(mixed_results["conclusion"].value_counts().to_string())
print()

ref = mixed_results["reference_subreddit"].iloc[0] if len(mixed_results) else "N/A"
print(f"Reference subreddit: {ref}")
print()

sig_mixed = mixed_results[mixed_results["significant"] == True].copy()
print(f"Significant (metric × subreddit) pairs: {len(sig_mixed)} / {len(mixed_results)}")
if len(sig_mixed):
    print(sig_mixed[["metric", "subreddit", "reference_subreddit",
                     "delta", "se_delta", "p_value", "p_value_bh", "conclusion"]]
          .sort_values(["metric", "subreddit"])
          .to_string(index=False))

mixed_results

In [ ]:
# β₂ Control Coefficients — Mixed-Effects Model
# β₂ is the same for every subreddit row within a given metric (it is estimated
# jointly across subreddits). We deduplicate to one row per metric for display.
print("=== Mixed-Effects — β₂ Control Coefficients ===\n")

_ctrl_names = ["post_depth", "edited", "score", "num_direct_replies", "controversiality"]
_b2_cols  = [f"beta2_{c}"    for c in _ctrl_names if f"beta2_{c}"    in mixed_results.columns]
_se_cols  = [f"se_beta2_{c}" for c in _ctrl_names if f"se_beta2_{c}" in mixed_results.columns]
_avail    = [c for c in _ctrl_names if f"beta2_{c}" in mixed_results.columns]

if _b2_cols:
    _b2 = (
        mixed_results[["metric"] + _b2_cols + _se_cols]
        .drop_duplicates(subset="metric")          # β₂ is metric-level, not subreddit-level
        .rename(columns={f"beta2_{c}": c for c in _avail}
                | {f"se_beta2_{c}": f"se_{c}" for c in _avail})
        .reset_index(drop=True)
    )
    print(_b2.to_string(index=False))
else:
    print("β₂ columns not found — re-run after updating regressions.py.")